# Laboratorio 10.2 — Workflow de agente con LangGraph

Ensamblaremos un flujo con clasificador, router, nodos RAG/herramienta/respuesta directa y validación Pydantic, reutilizando el grafo del paquete.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Ejecutar un workflow LangGraph con tres rutas condicionales.
- Validar su salida con un contrato Pydantic antes de entregarla.
- Inspeccionar evidencia documental y resultados de herramientas en el estado.
- Pausar antes de una herramienta y reanudar un hilo persistido tras aprobación.

## 2. Conceptos

Un workflow combina routing y nodos especializados con límites claros. La ruta RAG recupera documentos; la ruta herramienta valida y ejecuta una función; la ruta directa evita acciones innecesarias. La validación final convierte errores estructurales en fallos visibles. Checkpoint conserva progreso, mientras human-in-the-loop permite revisar operaciones antes de continuarlas.

## 3. Arquitectura

```mermaid
flowchart TD
    A[START] --> B[Clasificador de solicitud]
    B --> C{Router condicional}
    C --> D[RAG y busqueda FAQ]
    C --> E[Herramienta Pydantic]
    C --> F[Respuesta directa]
    D --> G[Validacion de salida]
    E --> G
    F --> G
    G --> H[END]
    E -. pausa/checkpoint .-> I[Aprobacion humana]
```

## 4. Preparación del entorno

El constructor y la lógica de nodos viven en `src/llm_engineering/agents/graph.py`. Aquí ejecutamos el workflow y medimos resultados; no se necesita proveedor ni credencial de LLM.

In [ ]:
import pandas as pd
from langgraph.checkpoint.memory import MemorySaver

from llm_engineering.environment import preparar_entorno
from llm_engineering.agents.graph import build_workflow
from llm_engineering.evaluation.metrics import classification_accuracy, schema_compliance_rate

raiz_curso = preparar_entorno()
workflow = build_workflow()
print('Nodos del workflow:', list(workflow.get_graph().nodes))

## 5. Definición del problema

Enrutaremos preguntas de devolución a FAQ, operaciones explícitas a la calculadora y saludos a respuesta directa. Un caso adicional pasa argumentos de clima explícitamente para demostrar la ruta de herramienta sin ambigüedad de lenguaje natural.

## 6. Línea base

La línea base invoca el clasificador por reglas del grafo y conserva su ruta. La salida se valida después del nodo especializado; las decisiones simuladas son reproducibles y no representan precisión de un LLM.

In [ ]:
solicitudes = [
    {'request': 'devolucion 30 dias', 'ruta_esperada': 'rag'},
    {'request': 'calcula 2 + 3', 'ruta_esperada': 'tool'},
    {'request': 'hola, necesito ayuda', 'ruta_esperada': 'direct'},
    {
        'request': 'consulta clima',
        'tool_name': 'clima_mock',
        'tool_arguments': {'city': 'Santiago'},
        'ruta_esperada': 'tool',
    },
]
print('Casos de prueba:', len(solicitudes))

## 7. Implementación

Invocamos cada entrada por separado. Para la ruta RAG registramos IDs de fuente; para las otras rutas, la respuesta Pydantic lleva resultado de herramienta o salida directa. Observa la misma forma externa con contenido específico por ruta.

In [ ]:
filas = []
for solicitud in solicitudes:
    resultado = workflow.invoke({
        clave: valor for clave, valor in solicitud.items() if clave != 'ruta_esperada'
    })
    salida = resultado['validated_output']
    filas.append({
        'request': solicitud['request'],
        'ruta_esperada': solicitud['ruta_esperada'],
        'ruta': salida['route'],
        'status': salida['status'],
        'answer': salida['answer'],
        'source_ids': salida['source_ids'],
        'schema_valid': resultado['validation_error'] is None,
    })
tabla_workflow = pd.DataFrame(filas)
display(tabla_workflow)

## 8. Experimentos

Mantenemos la configuración del grafo fija y variamos la intención de entrada para visitar las tres rutas. Luego activamos una interrupción antes de la calculadora; reanudar el mismo `thread_id` equivale a continuar después de una aprobación simulada.

In [ ]:
workflow_con_revision = build_workflow(
    checkpointer=MemorySaver(),
    interrupt_before_tools=True,
)
configuracion_hilo = {'configurable': {'thread_id': 'calculo-pendiente-01'}}
pausado = workflow_con_revision.invoke(
    {'request': 'calcula 9 * 9'},
    configuracion_hilo,
)
print('Ruta pausada:', pausado['route'])
print('La herramienta se ejecutó antes de aprobación:', 'raw_output' in pausado)

APROBACION_HUMANA = True
if APROBACION_HUMANA:
    reanudado = workflow_con_revision.invoke(None, configuracion_hilo)
    print('Salida reanudada:', reanudado['validated_output'])

## 9. Evaluación

Medimos exactitud de ruta y cumplimiento estructural. El conteo de fuentes informa cobertura de evidencia, no corrección de respuesta. La invocación de demo no llama a modelos; la pausa humana ocurre antes de ejecutar la herramienta.

In [ ]:
metricas = pd.DataFrame([{
    'route_accuracy': classification_accuracy(
        tabla_workflow['ruta'].tolist(), tabla_workflow['ruta_esperada'].tolist()
    ),
    'schema_valid': schema_compliance_rate(tabla_workflow['schema_valid'].tolist()),
    'solicitudes': len(tabla_workflow),
    'respuestas_rag_con_fuentes': int(
        ((tabla_workflow['ruta'] == 'rag') & tabla_workflow['source_ids'].map(bool)).sum()
    ),
    'llamadas_llm': 0,
}])
display(metricas)
tabla_workflow.to_csv(raiz_curso / 'outputs' / '10_02_workflow_results.csv', index=False)
metricas.to_csv(raiz_curso / 'outputs' / '10_02_workflow_metrics.csv', index=False)

## 10. Discusión

El router del paquete es determinista y pedagógico; un sistema real necesitaría clasificador evaluado, reglas de abstención y límites de costo. El checkpointer conserva estado, no autoriza la acción. Aprobación humana debe vincularse a argumentos concretos y al mismo hilo que se reanuda.

## 11. Ejercicios

1. Añade una ruta de aclaración y define qué nodo la valida.
2. Incluye otra consulta RAG y compara IDs recuperados con una etiqueta previa.
3. Rechaza automáticamente herramientas fuera de una lista por usuario.
4. Cambia aprobación simulada a falso y verifica que el flujo no reanude.

## 12. Desafío

Añade un clasificador basado en modelo detrás de la misma interfaz, un conjunto de rutas etiquetado y un checkpointer persistente. Mide tool accuracy, calidad de retrieval, tasa de validación, pasos, latencia y llamadas, manteniendo la ejecución remota optativa.

## 13. Ideas clave

- StateGraph expresa routing, control y estado explícitos.
- RAG, herramientas y respuesta directa son nodos separados.
- Pydantic valida la salida antes de entregarla.
- Checkpoint e interrupción facilitan revisión, pero no reemplazan autorización.